# Analise Exploratoria de Dados (AED) e Machine Learning - NCR Ride Bookings

Este notebook explora o dataset de corridas, faz limpeza e feature engineering, e aplica modelos de:

- regressao para prever `Booking Value`;
- classificacao para prever `Booking Status`;
- clusterizacao com KMeans e DBSCAN.


## 1. Importacao de bibliotecas


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.cluster import DBSCAN, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42


## 2. Carregamento dos dados

A celula abaixo funciona tanto no Kaggle quanto localmente.


In [ ]:
local_candidates = [
    Path('data/raw/ncr_ride_bookings.csv'),
    Path('ncr_ride_bookings.csv'),
]

kaggle_candidates = list(Path('/kaggle/input').rglob('*.csv')) if Path('/kaggle/input').exists() else []
preferred_kaggle = [file for file in kaggle_candidates if 'ncr_ride_bookings' in file.name.lower()]

if preferred_kaggle:
    data_file = preferred_kaggle[0]
elif kaggle_candidates:
    data_file = kaggle_candidates[0]
else:
    data_file = next((file for file in local_candidates if file.exists()), None)

if data_file is None:
    raise FileNotFoundError(
        'Nao encontrei o dataset. Anexe ncr_ride_bookings.csv no Kaggle ou coloque o arquivo em data/raw/.'
    )

print(f'Arquivo carregado: {data_file}')
df = pd.read_csv(data_file)
display(df.head())
print('Shape:', df.shape)


## 3. Entendimento inicial dos dados


In [ ]:
print('Tipos de dados:')
display(df.dtypes)

print('
Valores ausentes:')
display(df.isnull().sum().sort_values(ascending=False))

print('
Estatisticas numericas:')
display(df.describe(include='all').transpose())


## 4. Limpeza e preparacao dos dados


In [ ]:
df = df.drop_duplicates().copy()

df['Booking ID'] = df['Booking ID'].astype(str).str.replace('"', '', regex=False)
df['Customer ID'] = df['Customer ID'].astype(str).str.replace('"', '', regex=False)
df['datetime'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'].astype(str), errors='coerce')

df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month

df['is_completed'] = (df['Booking Status'] == 'Completed').astype(int)

print('Shape apos limpeza:', df.shape)
display(df.head())


## 5. Analise exploratoria de dados


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.countplot(data=df, x='Booking Status', order=df['Booking Status'].value_counts().index, ax=axes[0, 0])
axes[0, 0].set_title('Distribuicao do status das corridas')
axes[0, 0].tick_params(axis='x', rotation=20)

sns.countplot(data=df, x='Vehicle Type', order=df['Vehicle Type'].value_counts().index, ax=axes[0, 1])
axes[0, 1].set_title('Distribuicao por tipo de veiculo')
axes[0, 1].tick_params(axis='x', rotation=20)

sns.histplot(data=df, x='Booking Value', bins=30, kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Distribuicao de Booking Value')

sns.histplot(data=df, x='Ride Distance', bins=30, kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Distribuicao de Ride Distance')

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
ride_by_hour = df['hour'].value_counts().sort_index()
sns.barplot(x=ride_by_hour.index, y=ride_by_hour.values, color='steelblue')
plt.title('Quantidade de corridas por hora')
plt.xlabel('Hora do dia')
plt.ylabel('Quantidade')
plt.show()

plt.figure(figsize=(8, 5))
correlation_df = df[['Booking Value', 'Ride Distance', 'Avg VTAT', 'Avg CTAT', 'Driver Ratings', 'Customer Rating', 'hour', 'day_of_week', 'month']].copy()
correlation_df = correlation_df.dropna(how='all', axis=1)
sns.heatmap(correlation_df.corr(numeric_only=True), annot=True, cmap='Blues', fmt='.2f')
plt.title('Mapa de correlacao entre variaveis numericas')
plt.show()


## 6. Regressao

Objetivo: prever `Booking Value`.


In [ ]:
regression_target = 'Booking Value'
regression_features = [
    'Ride Distance',
    'Avg VTAT',
    'Avg CTAT',
    'Driver Ratings',
    'Customer Rating',
    'hour',
    'day_of_week',
    'month',
    'Vehicle Type',
    'Payment Method',
    'Pickup Location',
    'Drop Location',
]

regression_df = df[regression_features + [regression_target]].dropna(subset=[regression_target]).copy()

X_reg = regression_df[regression_features]
y_reg = regression_df[regression_target]

numeric_features_reg = X_reg.select_dtypes(include='number').columns.tolist()
categorical_features_reg = X_reg.select_dtypes(exclude='number').columns.tolist()

preprocessor_reg = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
        ]), numeric_features_reg),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features_reg),
    ]
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

linear_model = Pipeline([
    ('preprocessor', preprocessor_reg),
    ('model', LinearRegression()),
])

rf_regressor = Pipeline([
    ('preprocessor', preprocessor_reg),
    ('model', RandomForestRegressor(n_estimators=120, random_state=RANDOM_STATE, n_jobs=-1)),
])

linear_model.fit(X_train_reg, y_train_reg)
rf_regressor.fit(X_train_reg, y_train_reg)

pred_linear = linear_model.predict(X_test_reg)
pred_rf = rf_regressor.predict(X_test_reg)

regression_results = pd.DataFrame([
    {'Modelo': 'Linear Regression', 'MAE': mean_absolute_error(y_test_reg, pred_linear), 'R2': r2_score(y_test_reg, pred_linear)},
    {'Modelo': 'Random Forest Regressor', 'MAE': mean_absolute_error(y_test_reg, pred_rf), 'R2': r2_score(y_test_reg, pred_rf)},
]).sort_values(by='R2', ascending=False)

display(regression_results)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(y_test_reg, pred_linear, alpha=0.4)
axes[0].plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--')
axes[0].set_title('Linear Regression: real vs previsto')
axes[0].set_xlabel('Valor real')
axes[0].set_ylabel('Valor previsto')

axes[1].scatter(y_test_reg, pred_rf, alpha=0.4, color='darkgreen')
axes[1].plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--')
axes[1].set_title('Random Forest: real vs previsto')
axes[1].set_xlabel('Valor real')
axes[1].set_ylabel('Valor previsto')

plt.tight_layout()
plt.show()


## 7. Classificacao

Objetivo: prever `Booking Status`.


In [ ]:
classification_target = 'Booking Status'
classification_features = [
    'Booking Value',
    'Ride Distance',
    'Avg VTAT',
    'Avg CTAT',
    'hour',
    'day_of_week',
    'month',
    'Vehicle Type',
    'Payment Method',
    'Pickup Location',
    'Drop Location',
]

classification_df = df[classification_features + [classification_target]].dropna(subset=[classification_target]).copy()

X_clf = classification_df[classification_features]
y_clf = classification_df[classification_target].astype(str)

numeric_features_clf = X_clf.select_dtypes(include='number').columns.tolist()
categorical_features_clf = X_clf.select_dtypes(exclude='number').columns.tolist()

preprocessor_clf = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
        ]), numeric_features_clf),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features_clf),
    ]
)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_clf,
)

log_model = Pipeline([
    ('preprocessor', preprocessor_clf),
    ('model', LogisticRegression(max_iter=1000)),
])

rf_classifier = Pipeline([
    ('preprocessor', preprocessor_clf),
    ('model', RandomForestClassifier(n_estimators=120, random_state=RANDOM_STATE, n_jobs=-1)),
])

log_model.fit(X_train_clf, y_train_clf)
rf_classifier.fit(X_train_clf, y_train_clf)

pred_log = log_model.predict(X_test_clf)
pred_rf_clf = rf_classifier.predict(X_test_clf)

classification_results = pd.DataFrame([
    {'Modelo': 'Logistic Regression', 'Accuracy': accuracy_score(y_test_clf, pred_log)},
    {'Modelo': 'Random Forest Classifier', 'Accuracy': accuracy_score(y_test_clf, pred_rf_clf)},
]).sort_values(by='Accuracy', ascending=False)

display(classification_results)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

class_labels = sorted(y_test_clf.unique())
cm_log = confusion_matrix(y_test_clf, pred_log, labels=class_labels)
cm_rf = confusion_matrix(y_test_clf, pred_rf_clf, labels=class_labels)

sns.heatmap(cm_log, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels, ax=axes[0])
axes[0].set_title('Matriz de confusao - Logistic Regression')
axes[0].set_xlabel('Previsto')
axes[0].set_ylabel('Real')

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', xticklabels=class_labels, yticklabels=class_labels, ax=axes[1])
axes[1].set_title('Matriz de confusao - Random Forest')
axes[1].set_xlabel('Previsto')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()


## 8. Clusterizacao com KMeans


In [ ]:
cluster_features = [
    'Booking Value',
    'Ride Distance',
    'Avg VTAT',
    'Avg CTAT',
    'Driver Ratings',
    'Customer Rating',
    'hour',
    'day_of_week',
    'month',
]

cluster_df = df[cluster_features].copy()
cluster_df = cluster_df.dropna(subset=['Booking Value', 'Ride Distance', 'Avg VTAT', 'Avg CTAT']).copy()

cluster_imputer = SimpleImputer(strategy='median')
cluster_scaled = StandardScaler().fit_transform(cluster_imputer.fit_transform(cluster_df))

kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(cluster_scaled)
cluster_df['kmeans_cluster'] = kmeans_labels

kmeans_silhouette = silhouette_score(cluster_scaled, kmeans_labels)
print('Silhouette Score do KMeans:', round(kmeans_silhouette, 4))
display(cluster_df['kmeans_cluster'].value_counts().sort_index())


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=cluster_df, x='Ride Distance', y='Booking Value', hue='kmeans_cluster', palette='tab10', alpha=0.7)
plt.title('Clusters do KMeans: distancia vs valor da corrida')
plt.show()


## 9. Clusterizacao com DBSCAN


In [ ]:
dbscan = DBSCAN(eps=0.9, min_samples=8)
dbscan_labels = dbscan.fit_predict(cluster_scaled)
cluster_df['dbscan_cluster'] = dbscan_labels

clusters_sem_ruido = len({label for label in dbscan_labels if label != -1})
ruidos = int((dbscan_labels == -1).sum())

print('Clusters encontrados pelo DBSCAN:', clusters_sem_ruido)
print('Pontos marcados como ruido:', ruidos)
display(cluster_df['dbscan_cluster'].value_counts().sort_index())


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=cluster_df, x='Ride Distance', y='Booking Value', hue='dbscan_cluster', palette='tab10', alpha=0.7)
plt.title('DBSCAN: clusters e ruido')
plt.show()


## 10. Conclusoes

Com este fluxo, o projeto mostra:

- uma parte exploratoria para entender o comportamento das corridas;
- regressao para prever o valor da corrida;
- classificacao para prever o status da reserva;
- clusterizacao para identificar perfis e padroes ocultos.

Na entrega, vale comentar quais modelos tiveram melhor desempenho e quais variaveis pareceram mais relevantes para explicar o comportamento das corridas.
